In [2]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=0
%pylab inline
%load_ext autoreload
%autoreload 2

env: CUDA_DEVICE_ORDER=PCI_BUS_ID
env: CUDA_VISIBLE_DEVICES=0
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

from ssl_neuron.datasets import GraphDataset
from ssl_neuron.utils import plot_neuron, neighbors_to_adjacency_torch, compute_eig_lapl_torch_batch
from ssl_neuron.graphdino import create_model

/gpfs01/berens/user/pgivehchian/ssl_neuron/.venv39/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
TRIAL = "trial5"

#### Load config

In [6]:
config = json.load(open(f'../configs/{TRIAL}.json'))

#### Load model + checkpoint

In [7]:
cell_meta = pd.read_csv(os.path.join(config['data']['path'], 'cell_meta.csv'))
cell_meta['cell_id'] = cell_meta['cell_id'].astype(str)

In [ ]:
model = create_model(config)

state_dict = torch.load(f'../ckpts/{TRIAL}.pt')

model.load_state_dict(state_dict)

In [13]:
model.eval()
model.cuda();

#### Load dataset

In [14]:
dset = GraphDataset(config, mode='all')

100%|██████████| 3690/3690 [04:59<00:00, 12.32it/s]


#### Run inference

In [18]:
latents = np.zeros((dset.num_samples, config['model']['dim']))

cell_meta['latent'] = pd.Series(
    [None] * len(cell_meta),
    index=cell_meta.index,
    dtype=object
)

cell_meta['cell_id'] = cell_meta['cell_id'].astype(str)

for i in tqdm(range(dset.num_samples)):
    feat, neigh, cell_id = dset.__getsingleitem__(i)
    cell_id = str(cell_id)

    adj = neighbors_to_adjacency_torch(
        neigh, list(neigh.keys())
    ).float().cuda()[None, :]

    lapl = compute_eig_lapl_torch_batch(
        adj,
        pos_enc_dim=config['model']['pos_dim']
    ).float().cuda()

    feat = torch.from_numpy(feat).float().cuda()[None, :]

    with torch.no_grad():
        embedding = model.student_encoder.forward(feat, adj, lapl)[0]
        embedding = embedding.cpu().numpy().squeeze()

    latents[i] = embedding

    mask = cell_meta['cell_id'] == cell_id

    for idx in cell_meta.index[mask]:
        cell_meta.at[idx, 'latent'] = embedding

print("\nEmbedding extraction complete!")
print("Latents matrix shape:", latents.shape)
print("Rows with embeddings:", cell_meta['latent'].notna().sum())


100%|██████████| 3690/3690 [04:02<00:00, 15.24it/s]


Embedding extraction complete!
Latents matrix shape: (3690, 32)
Rows with embeddings: 3690


In [19]:
z_means = []

for _, row in tqdm(cell_meta.iterrows(), total=len(cell_meta)):

    cell_id = int(row['cell_id'])

    feature_path = os.path.join(
        config['data']['path'],
        'skeletons',
        str(cell_id),
        'features.npy'
    )

    if os.path.exists(feature_path):

        features = np.load(feature_path)
        z_values = features[:, 2]
        z_mean = np.nanmean(z_values)

    else:
        print(f"Missing features: {cell_id}")
        z_mean = np.nan

    z_means.append(z_mean)

cell_meta['z_mean'] = z_means

100%|██████████| 3690/3690 [00:02<00:00, 1568.21it/s]


In [20]:
cell_meta.to_csv(
    os.path.join(config['trainer']['latents_dir'], f'{TRIAL}.csv'),
    index=False
)